# FabricQA – Fabric Defect Classifier (SE5070 Enterprise Mobility)

Trains the defect classifier used by the FabricQA app on three data sources:

- **Fabric Defect Dataset** – `rmshashi/fabric-defect-dataset` (hole, horizontal, vertical) – University of Moratuwa Intelligence Lab
- **Fabric Stain Dataset** – `priemshpathirana/fabric-stain-dataset` (stain, defect-free) – University of Moratuwa Intelligence Lab
- **FabricQA own photos** – real-world phone photos taken by the author, one folder per class (`defect_free`, `hole`, `horizontal`, `stain`, `vertical`)

**Before running (Kaggle notebook settings, right panel):**
1. **Add Input** → add all three datasets.
2. **Session options → Accelerator** → GPU T4 x2.
3. **Session options → Internet** → On.
4. Run with **Save Version → Save & Run All (Commit)** so the output files are kept.

**Changes from v2:** own phone photos added (EXIF orientation fixed, resized), a separate real-world test set taken only from own photos, own training photos repeated so they aren't drowned out by the public data, evaluation reported separately for the public and real-world test sets, and the model version and thresholds written to `labels.json`.

In [ ]:
import os, json, random, hashlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

INPUT_DIR  = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'
TMP_DIR    = '/tmp'
OWN_KEYWORD = 'fabricqa-own'   # any input path containing this is treated as an own photo
OWN_PREP_DIR = f'{TMP_DIR}/own_prepared'
OWN_REPEAT = 3                 # each own training photo is used this many times per epoch
OWN_MAX_SIDE = 1024            # own photos are resized to this before training

MODEL_VERSION = '1.1'
IMG_SIZE   = 224
BATCH_SIZE = 32
HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 10
UNFREEZE_LAST_N = 40
CONFIDENCE_THRESHOLD = 0.70    # below this the app shows "needs manual check"
DEFECT_FREE_THRESHOLD = 0.85   # stricter threshold before trusting a "defect_free" verdict

print('TensorFlow', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 1. Find the images and label them from their folder names

All inputs are walked automatically. Mask images are skipped. The label comes from keywords in the file path, checked in order (first match wins).

- The defect dataset's vertical folder is spelled **`verticle`**, so both spellings are matched.
- **`captured/Lines`** (uncropped swatches with faint lines in mixed orientations) is excluded.
- Own photos are recognised by `fabricqa-own` in their path. Their folders must be named exactly `defect_free`, `hole`, `horizontal`, `stain`, `vertical`.

In [ ]:
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')
SKIP_KEYWORDS = ['mask']
LINES_LABEL = None            # None = exclude captured/Lines

LABEL_RULES = [
    ('defect_free', ['defect free', 'defect_free', 'defect-free', 'non defect', 'non_defect', 'nodefect', 'no defect', 'good']),
    ('hole',        ['hole']),
    ('horizontal',  ['horizontal']),
    ('vertical',    ['vertical', 'verticle']),
    ('stain',       ['stain']),
]

def label_for(path):
    p = path.lower().replace('\\', '/')
    if any(k in p for k in SKIP_KEYWORDS):
        return None
    if '/captured/lines' in p:
        return LINES_LABEL
    for label, keys in LABEL_RULES:
        if any(k in p for k in keys):
            return label
    return 'UNMATCHED'

raw_paths, raw_labels, unmatched_dirs, heic_files = [], [], Counter(), 0
for root, _, files in os.walk(INPUT_DIR):
    for f in files:
        if f.lower().endswith(('.heic', '.heif')):
            heic_files += 1
            continue
        if not f.lower().endswith(IMG_EXT):
            continue
        full = os.path.join(root, f)
        lab = label_for(full)
        if lab is None:
            continue
        if lab == 'UNMATCHED':
            unmatched_dirs[root] += 1
            continue
        raw_paths.append(full); raw_labels.append(lab)

is_own_raw = [OWN_KEYWORD in p.lower() for p in raw_paths]
print('Images per class (before de-duplication):')
print(f'  {"class":12s} {"public":>7s} {"own":>5s}')
for c in sorted(set(raw_labels)):
    pub = sum(1 for l, o in zip(raw_labels, is_own_raw) if l == c and not o)
    own = sum(1 for l, o in zip(raw_labels, is_own_raw) if l == c and o)
    print(f'  {c:12s} {pub:7d} {own:5d}')
print('Total:', len(raw_paths), f'(own photos: {sum(is_own_raw)})')
if heic_files:
    print(f'\nWARNING: {heic_files} HEIC/HEIF files were skipped. Re-export them as JPEG '
          '(iPhone: Settings → Camera → Formats → Most Compatible).')
if sum(is_own_raw) == 0:
    print('\nWARNING: no own photos found - is the fabricqa-own dataset added as an Input?')
if unmatched_dirs:
    print('\nUnmatched folders (not used):')
    for d, n in unmatched_dirs.most_common(10):
        print(f'  {n:5d}  {d}')

### Remove exact duplicates, then prepare own photos

Duplicates are removed by file hash before splitting so identical images can't land in both train and test.

Phone photos store their rotation in EXIF metadata, which TensorFlow ignores when decoding. Without fixing it, a portrait photo is read sideways – which would turn a **horizontal** line into a **vertical** one. Own photos are therefore rotated upright, **centre-cropped to a square (exactly as the app's camera screen crops before sending)**, and resized to at most 1024 px before training.

In [ ]:
by_hash = {}
for p, l in zip(raw_paths, raw_labels):
    h = hashlib.md5(open(p, 'rb').read()).hexdigest()
    by_hash.setdefault(h, []).append((p, l))

paths, labels, conflicts = [], [], 0
for items in by_hash.values():
    if len({l for _, l in items}) > 1:
        conflicts += 1
        continue
    paths.append(items[0][0]); labels.append(items[0][1])
print(f'Removed {len(raw_paths) - len(paths)} duplicate images ({conflicts} conflicting groups dropped)')

def prepare_own(path, label):
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    # Centre square crop - the same crop the app's camera screen makes before sending a photo
    side = min(img.size)
    left, top = (img.width - side) // 2, (img.height - side) // 2
    img = img.crop((left, top, left + side, top + side))
    img.thumbnail((OWN_MAX_SIDE, OWN_MAX_SIDE))
    out_dir = os.path.join(OWN_PREP_DIR, label)
    os.makedirs(out_dir, exist_ok=True)
    out = os.path.join(out_dir, hashlib.md5(path.encode()).hexdigest()[:12] + '.jpg')
    img.save(out, 'JPEG', quality=92)
    return out

is_own = []
for i, (p, l) in enumerate(zip(paths, labels)):
    own = OWN_KEYWORD in p.lower()
    is_own.append(own)
    if own:
        paths[i] = prepare_own(p, l)
is_own = np.array(is_own)

print('Images per class (final):')
for c in sorted(set(labels)):
    pub = sum(1 for l, o in zip(labels, is_own) if l == c and not o)
    own = sum(1 for l, o in zip(labels, is_own) if l == c and o)
    print(f'  {c:12s} public {pub:4d}   own {own:3d}')
print('Total:', len(paths), f'(own: {is_own.sum()})')

In [ ]:
# Peek at examples per class – own photos in the right-hand columns
CLASS_NAMES = sorted(set(labels))
fig, axes = plt.subplots(len(CLASS_NAMES), 6, figsize=(16, 3 * len(CLASS_NAMES)))
for r, cls in enumerate(CLASS_NAMES):
    pub = [p for p, l, o in zip(paths, labels, is_own) if l == cls and not o]
    own = [p for p, l, o in zip(paths, labels, is_own) if l == cls and o]
    picks = random.sample(pub, k=min(3, len(pub))) + random.sample(own, k=min(3, len(own)))
    for c in range(6):
        ax = axes[r, c]; ax.axis('off')
        if c < len(picks):
            ax.imshow(plt.imread(picks[c]))
            ax.set_title(f'{cls} ({"own" if OWN_PREP_DIR in picks[c] else "public"})', fontsize=9)
plt.tight_layout(); plt.show()

## 2. Split the data

- **Public images:** 70 / 15 / 15 train / validation / test, stratified.
- **Own photos:** 60 / 15 / 25, stratified. The 25% held out is the **real-world test set** – the closest measure of how the app will perform on phone photos.
- Own training photos are repeated `OWN_REPEAT` times so ~90 phone photos aren't drowned out by ~700 public images. Augmentation makes each repeat look slightly different.

In [ ]:
CLASS_NAMES = sorted(set(labels))
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
X = np.array(paths)
y = np.array([class_to_idx[l] for l in labels])

def safe_split(X, y, test_size):
    strat = y if min(Counter(y).values()) >= 2 else None
    return train_test_split(X, y, test_size=test_size, stratify=strat, random_state=SEED)

Xp, yp = X[~is_own], y[~is_own]
Xp_train, Xp_tmp, yp_train, yp_tmp = safe_split(Xp, yp, 0.30)
Xp_val, Xp_test, yp_val, yp_test = safe_split(Xp_tmp, yp_tmp, 0.50)

Xo, yo = X[is_own], y[is_own]
if len(Xo) >= 10:
    Xo_trval, Xo_test, yo_trval, yo_test = safe_split(Xo, yo, 0.25)
    Xo_train, Xo_val, yo_train, yo_val = safe_split(Xo_trval, yo_trval, 0.20)
else:
    Xo_train = Xo_val = Xo_test = np.array([], dtype=str)
    yo_train = yo_val = yo_test = np.array([], dtype=int)
    print('Fewer than 10 own photos - training on public data only.')

X_train = np.concatenate([Xp_train] + [Xo_train] * OWN_REPEAT)
y_train = np.concatenate([yp_train] + [yo_train] * OWN_REPEAT)
X_val = np.concatenate([Xp_val, Xo_val]); y_val = np.concatenate([yp_val, yo_val])

print(f'Public: train {len(Xp_train)} | val {len(Xp_val)} | test {len(Xp_test)}')
print(f'Own:    train {len(Xo_train)} (x{OWN_REPEAT}) | val {len(Xo_val)} | test {len(Xo_test)}')
print(f'Training set size per epoch: {len(X_train)}')

cw = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=y_train)
CLASS_WEIGHTS = {i: float(w) for i, w in enumerate(cw)}
print('Class weights:', {CLASS_NAMES[i]: round(w, 2) for i, w in CLASS_WEIGHTS.items()})

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.float32), label   # raw 0-255; scaling happens inside the model

def make_ds(X, y, training):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(len(X), seed=SEED)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(X_train, y_train, True)
val_ds   = make_ds(X_val, y_val, False)

## 3. Model – MobileNetV2 transfer learning

Small and fast, suiting a mobile-first app and a lightweight inference server. Pixel scaling lives inside the model, so it accepts raw 0–255 RGB images resized to 224×224. Augmentation is only active during training. Flips and small rotations never turn a horizontal line into a vertical one.

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
], name='augmentation')

base = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image')
x = augment(inputs)
x = tf.keras.layers.Rescaling(1/127.5, offset=-1, name='scale_to_minus1_1')(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
x = tf.keras.layers.Dropout(0.3, name='dropout')(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax', name='defect_probs')(x)
model = tf.keras.Model(inputs, outputs, name='fabricqa_defect_classifier')

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 4. Train the classification head

In [ ]:
def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2),
    ]

hist1 = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
                  class_weight=CLASS_WEIGHTS, callbacks=make_callbacks())

P1_WEIGHTS = f'{TMP_DIR}/phase1.weights.h5'
model.save_weights(P1_WEIGHTS)
p1_best = min(hist1.history['val_loss'])
print(f'Phase 1 best val_loss: {p1_best:.4f}')

## 5. Fine-tune the top of MobileNetV2

Only the last `UNFREEZE_LAST_N` layers are unfrozen and **BatchNormalization layers stay frozen** (unfreezing them in v1 made training loss jump from 0.37 to 2.59). If fine-tuning doesn't beat phase 1, the phase-1 weights are restored.

In [ ]:
base.trainable = True
for layer in base.layers[:-UNFREEZE_LAST_N]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
print('Trainable layers in base:', sum(l.trainable for l in base.layers))

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
                  class_weight=CLASS_WEIGHTS, callbacks=make_callbacks())

p2_best = min(hist2.history['val_loss'])
if p2_best > p1_best:
    model.load_weights(P1_WEIGHTS)
    FINAL_PHASE = 'head only (fine-tuning did not improve val_loss)'
else:
    FINAL_PHASE = 'fine-tuned'
print(f'Phase 2 best val_loss: {p2_best:.4f} -> using {FINAL_PHASE} weights')

In [ ]:
def merge(h1, h2, key):
    return h1.history.get(key, []) + h2.history.get(key, [])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, key in enumerate(['accuracy', 'loss']):
    ax[i].plot(merge(hist1, hist2, key), label='train')
    ax[i].plot(merge(hist1, hist2, 'val_' + key), label='val')
    ax[i].axvline(len(hist1.history['loss']) - 0.5, ls='--', c='grey', label='fine-tune starts')
    ax[i].set_title(key); ax[i].set_xlabel('epoch'); ax[i].legend()
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150); plt.show()

## 6. Evaluate – public test set and real-world test set

Each test set is evaluated separately. The **real-world (own photos) result is the honest estimate** of how the app performs on phone photos. The threshold rule matches the inference server: 0.70 for defects, 0.85 before trusting "defect_free".

In [ ]:
def required_threshold(pred_idx):
    return np.where(pred_idx == class_to_idx.get('defect_free', -1), DEFECT_FREE_THRESHOLD, CONFIDENCE_THRESHOLD)

def evaluate(name, slug, X_eval, y_eval):
    if len(X_eval) == 0:
        print(f'{name}: no images'); return None
    prob = model.predict(make_ds(X_eval, y_eval, False), verbose=0)
    pred = prob.argmax(axis=1)
    acc = float((pred == y_eval).mean())
    print(f'\n=== {name}: accuracy {acc:.4f} on {len(y_eval)} images ===')
    print(classification_report(y_eval, pred, labels=range(len(CLASS_NAMES)),
                                target_names=CLASS_NAMES, digits=3, zero_division=0))

    cm = confusion_matrix(y_eval, pred, labels=range(len(CLASS_NAMES)))
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha='right')
    ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(f'Confusion matrix – {name}')
    plt.colorbar(im); plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/confusion_matrix_{slug}.png', dpi=150); plt.show()

    conf = prob.max(axis=1)
    keep = conf >= required_threshold(pred)
    acc_kept = float((pred[keep] == y_eval[keep]).mean()) if keep.any() else float('nan')
    print(f'With the app thresholds: {keep.mean():.1%} auto-accepted at {acc_kept:.1%} accuracy, '
          f'{(~keep).sum()} sent to manual check')
    df = class_to_idx.get('defect_free')
    if df is not None:
        missed = (pred == df) & (y_eval != df)
        print(f'Real defects predicted as defect_free: {missed.sum()} of {(y_eval != df).sum()} '
              f'({(missed & keep).sum()} not caught by the threshold)')
    return {'accuracy': round(acc, 4), 'auto_accepted': round(float(keep.mean()), 4),
            'accuracy_when_accepted': round(acc_kept, 4), 'n': int(len(y_eval))}

results = {
    'public_test': evaluate('Public test set', 'public', Xp_test, yp_test),
    'real_world_test': evaluate('Real-world test set (own photos)', 'own', Xo_test, yo_test),
}

## 7. Export for the app

The export model shares the trained weights but drops the augmentation layers (which TFLite can't convert). `labels.json` carries the classes, thresholds, model version and results – the inference server reads its settings from it, so updating the model only means replacing these two files.

In [ ]:
serve_in = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image')
x = model.get_layer('scale_to_minus1_1')(serve_in)
x = base(x, training=False)
x = model.get_layer('gap')(x)
export_model = tf.keras.Model(serve_in, model.get_layer('defect_probs')(x), name='fabricqa_defect_classifier_serving')

check = next(iter(val_ds))[0][:8]
assert np.allclose(model.predict(check, verbose=0), export_model.predict(check, verbose=0), atol=1e-4)

export_model.save(f'{OUTPUT_DIR}/fabric_defect_classifier.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
with open(f'{OUTPUT_DIR}/fabric_defect_classifier.tflite', 'wb') as f:
    f.write(converter.convert())

with open(f'{OUTPUT_DIR}/labels.json', 'w') as f:
    json.dump({'classes': CLASS_NAMES, 'img_size': IMG_SIZE, 'input': 'RGB 0-255',
               'model_version': MODEL_VERSION,
               'confidence_threshold': CONFIDENCE_THRESHOLD,
               'defect_free_threshold': DEFECT_FREE_THRESHOLD,
               'weights': FINAL_PHASE,
               'results': results}, f, indent=2)

print(open(f'{OUTPUT_DIR}/labels.json').read())

## 8. Try a single real-world prediction

In [ ]:
def predict_image(path):
    img, _ = load_image(path, 0)
    probs = export_model.predict(tf.expand_dims(img, 0), verbose=0)[0]
    top = int(np.argmax(probs))
    need = DEFECT_FREE_THRESHOLD if CLASS_NAMES[top] == 'defect_free' else CONFIDENCE_THRESHOLD
    label = CLASS_NAMES[top] if probs[top] >= need else 'needs_manual_check'
    return label, float(probs[top]), {c: round(float(p), 3) for c, p in zip(CLASS_NAMES, probs)}

pool = list(Xo_test) if len(Xo_test) else list(Xp_test)
sample_path = random.choice(pool)
label, conf_top, all_probs = predict_image(sample_path)
plt.imshow(plt.imread(sample_path)); plt.axis('off')
plt.title(f'Result: {label} (top confidence {conf_top:.1%})'); plt.show()
print('Actual class:', CLASS_NAMES[y[list(X).index(sample_path)]])
print(all_probs)

In [ ]:
# Files to download from the Output tab
!ls -lh /kaggle/working